# 02 — State Tracker prototype (100-session smoke)

Per RecSys_Challenge_Plan §A1 + §8 file list. Validates the `StateTracker` parse-validity at scale before W2 CMQR depends on it.

**What this notebook does:**
1. Mounts Drive (HF datasets cache) and installs minimal deps (torch + transformers + datasets).
2. Loads Qwen-2.5-1.5B-Instruct on the A100 GPU.
3. Runs `scripts/smoke_state_tracker.py --n-sessions 100` (~800 (session, turn) extractions).
4. Saves the resulting `data/smoke_state_tracker_report.json` to Drive.
5. Prints `parse_validity_excl_cache` prominently so the user can read it without scrolling.

**Gate (plan §A1):** `parse_validity_excl_cache` ≥ 0.99 → ship as-is. 0.95–0.99 → ship with caveat. <0.95 → investigate prompt/model.

**Gap-4 context:** local MPS smoke ran 25 sessions in ~6.7 min. 100 sessions ≈ 27 min on MPS (too slow); 100 sessions on A100 should finish in 2-4 min.

**Wall time on A100:** target 3-5 min including model load.

In [ ]:
# 1) Verify GPU.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model branch.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026
print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%ndate:    %ai%nsubject: %s'
print()

In [ ]:
# 2b) Mount Drive + wire persistent caches.
# Same convention as 029/030: persist HF datasets + experiments/cache,
# do NOT persist HF model weights (Drive read slower than HF download).
import os, shutil
from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print(f'first mount attempt failed: {e}\nretrying with force_remount=True ...')
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    drive.mount('/content/drive', force_remount=True)

assert os.path.isdir('/content/drive/MyDrive'), (
    'Drive mount failed — /content/drive/MyDrive does not exist. '
    'Common fixes: complete the OAuth popup fully, disable popup '
    'blocker, or sign in to Google in this browser tab on the same account.'
)

DRIVE_BASE = '/content/drive/MyDrive/recsys2026-cache'
for d in [f'{DRIVE_BASE}/hf_datasets', f'{DRIVE_BASE}/experiments_cache',
          f'{DRIVE_BASE}/state_tracker_smoke']:
    os.makedirs(d, exist_ok=True)

os.environ['HF_DATASETS_CACHE'] = f'{DRIVE_BASE}/hf_datasets'
%env HF_DATASETS_CACHE={DRIVE_BASE}/hf_datasets

EXPECTED_CACHE = '/content/recsys2026/music-crs-baselines/experiments/cache'
os.makedirs(os.path.dirname(EXPECTED_CACHE), exist_ok=True)
if os.path.exists(EXPECTED_CACHE) and not os.path.islink(EXPECTED_CACHE):
    shutil.rmtree(EXPECTED_CACHE)
if not os.path.islink(EXPECTED_CACHE):
    os.symlink(f'{DRIVE_BASE}/experiments_cache', EXPECTED_CACHE)

print(f'datasets cache  : {os.environ["HF_DATASETS_CACHE"]}')
print(f'experiments dir : {EXPECTED_CACHE} -> {os.readlink(EXPECTED_CACHE)}')

In [ ]:
# 3) Install minimal deps. The smoke script imports torch + transformers + datasets + tqdm + pandas.
# Avoid the full requirements.txt — the smoke needs neither bm25 nor the retrieval libs.
!pip install -q --upgrade transformers datasets pandas tqdm
!python -c 'import torch, transformers; print("torch", torch.__version__, "cuda", torch.cuda.is_available())'

In [ ]:
# 4) Smoke parameters.
N_SESSIONS = 100
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
DEVICE = 'cuda'
MAX_NEW_TOKENS = 96
SEED = 42

In [ ]:
# 5) Run the smoke. Uses the existing scripts/smoke_state_tracker.py — no inline
# duplication. The smoke prints a summary block + writes data/smoke_state_tracker_report.json.
!nvidia-smi --query-gpu=memory.total,memory.used,memory.free --format=csv
!cd /content/recsys2026 && python scripts/smoke_state_tracker.py \
    --n-sessions {N_SESSIONS} \
    --seed {SEED} \
    --model {MODEL_NAME} \
    --device {DEVICE} \
    --max-new-tokens {MAX_NEW_TOKENS}

In [ ]:
# 6) Read report + print parse_validity prominently.
import json
from pathlib import Path
report_path = Path('/content/recsys2026/data/smoke_state_tracker_report.json')
assert report_path.exists(), f'report not found at {report_path}'
with report_path.open() as f:
    report = json.load(f)

stats = report.get('stats', {})
pv = stats.get('parse_validity_excl_cache', 0.0)

print('=' * 60)
print('STATE TRACKER 100-SESSION SMOKE — KEY RESULTS')
print('=' * 60)
print(f'  parse_validity_excl_cache : {pv:.4f}')
print(f'  ok_first_try              : {stats.get("ok_first_try", 0)}')
print(f'  ok_after_retry            : {stats.get("ok_after_retry", 0)}')
print(f'  fallback_to_prior         : {stats.get("fallback_to_prior", 0)}')
print(f'  drop                      : {stats.get("drop", 0)}')
print(f'  total calls               : {stats.get("calls", 0)}')
print(f'  cache hits                : {stats.get("cache_hits", 0)}')
print(f'  per-call mean ms          : {report.get("per_call_mean_ms")}')
print(f'  total elapsed (s)         : {report.get("elapsed_seconds")}')
print()
print('GATE (plan §A1):')
if pv >= 0.99:
    print(f'  PASS    {pv:.4f} >= 0.99 — ship as-is')
elif pv >= 0.95:
    print(f'  PASS*   0.95 <= {pv:.4f} < 0.99 — ship with caveat')
else:
    print(f'  FAIL    {pv:.4f} < 0.95 — investigate prompt/model')
print('=' * 60)

In [ ]:
# 7) Save report to Drive so it persists past the Colab runtime.
import shutil
drive_dst = f'{DRIVE_BASE}/state_tracker_smoke/smoke_state_tracker_report_n{N_SESSIONS}.json'
shutil.copy(report_path, drive_dst)
print(f'report saved to: {drive_dst}')
!ls -lh {DRIVE_BASE}/state_tracker_smoke/

In [ ]:
# 8) Failure inspection (only if any failed).
fails = report.get('stats', {}).get('drop', 0) + report.get('stats', {}).get('fallback_to_prior', 0)
if fails == 0:
    print('No parse failures observed across all 100 sessions. State tracker is production-ready.')
else:
    print(f'{fails} parse-failure events. Re-run with --debug-failures to capture raw outputs.')
    print('Open data/smoke_state_tracker_report.json for the per-call breakdown.')

# Sample a few extracted states to eyeball the quality.
print('\n=== Sample extracted states (first 6) ===')
for s in report.get('samples', []):
    print(f'\n  session={s["session_id"]}  turn={s["turn"]}  ms={s["ms"]}')
    print(f'  query: {s["user_query"][:80]!r}')
    print(f'  state: {s["state"]}')

## After the smoke

If `parse_validity_excl_cache >= 0.99`:
- Mark the W1 §A1 gate PASS in `documents/RecSys_Challenge_Plan.md`.
- Commit the report at `data/smoke_state_tracker_report.json` to git (it's small).
- Proceed to W2 (CMQR + state-conditioned retrieval).

If `0.95 <= pv < 0.99`:
- Document the failures in `data/reward_design_critique.md` (Gap 7 artifact).
- Pre-cache state for failure cases via the existing `_session_last_state` fallback so production never sees a drop.

If `pv < 0.95`:
- Examine sample failure outputs (re-run with `--debug-failures` flag if needed).
- Iterate on `music-crs-baselines/mcrs/system_prompts/state_extraction.txt` (more directive few-shots, stronger 'OUTPUT ONLY' framing) before retrying.